In [78]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [ ]:
batch_size = 64
sequence_len = 64
embedding_size = 32
head_count = 4
layer_count = 4
steps = 5000

In [80]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

# train stays on cpu (unfold is a view, moving it would copy every window), batches move to device
x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len).to(device=device)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len).to(device=device)

In [81]:
# module declaration
class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_size, sequence_len):
        super().__init__()

        self.token_embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.position_embedding = nn.Embedding(num_embeddings=sequence_len, embedding_dim=embedding_size)

    def forward(self, input):
        return self.token_embedding(input=input) + self.position_embedding(torch.arange(input.size(dim=1), device=input.device))

class SelfAttention(nn.Module):
    def __init__(self, embedding_size, head_size, sequence_len):
        super().__init__()

        self.head_size = head_size
        self.query = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)
        self.key = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)
        self.value = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)

        tril = torch.tril(input=torch.ones(size=(sequence_len, sequence_len), dtype=torch.bool))
        self.register_buffer(name="mask", tensor=tril, persistent=False)

    def forward(self, input):
        query = self.query(input=input)
        key = self.key(input=input)
        value = self.value(input=input)

        T = input.size(dim=1)

        scores = query @ key.transpose(dim0=-2, dim1=-1) * self.head_size ** -0.5
        scores = scores.masked_fill(mask=~self.mask[:T, :T], value=float("-inf"))
        weights = torch.softmax(input=scores, dim=-1)

        return weights @ value

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_size, head_count, sequence_len):
        super().__init__()
        self.heads = nn.ModuleList(
            [SelfAttention(embedding_size=embedding_size, head_size=embedding_size // head_count, sequence_len=sequence_len) for _ in range(head_count)]
        )
        self.projection = nn.Linear(in_features=embedding_size, out_features=embedding_size)

    def forward(self, input):
        head_outputs = []
        for head in self.heads:
            head_output = head(input=input)
            head_outputs.append(head_output)
        return self.projection(input=torch.cat(head_outputs, dim=-1))

class FeedForward(nn.Module):
    def __init__(self, embedding_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=embedding_size, out_features=4 * embedding_size),
            nn.ReLU(),
            nn.Linear(in_features=4 * embedding_size, out_features=embedding_size),
        )
        
    def forward(self, input):
        return self.layers(input=input)

class Block(nn.Module):
    def __init__(self, embedding_size, head_count, sequence_len):
        super().__init__()
        self.attention = MultiHeadAttention(embedding_size=embedding_size, head_count=head_count, sequence_len=sequence_len)
        self.layer_norm1 = nn.LayerNorm(normalized_shape=embedding_size)
        self.feed_forward = FeedForward(embedding_size=embedding_size)
        self.layer_norm2 = nn.LayerNorm(normalized_shape=embedding_size)

    def forward(self, input):
        input = input + self.attention(input=self.layer_norm1(input=input))
        input = input + self.feed_forward(input=self.layer_norm2(input=input))
        return input

In [82]:
# model declaration
class Transformer(nn.Module):
    def __init__(
        self, vocab_size, embedding_size, sequence_len, head_count, layer_count
    ):
        super().__init__()
        self.layers = nn.Sequential(
            TokenAndPositionEmbedding(
                vocab_size=vocab_size,
                embedding_size=embedding_size,
                sequence_len=sequence_len,
            ),
            nn.Sequential(
                *[
                    Block(
                        embedding_size=embedding_size,
                        head_count=head_count,
                        sequence_len=sequence_len,
                    )
                    for _ in range(layer_count)
                ]
            ),
            nn.LayerNorm(normalized_shape=embedding_size),
            nn.Linear(in_features=embedding_size, out_features=vocab_size),
        )

    def forward(self, input):
        return self.layers(input=input)

In [83]:
# model initialization
model = Transformer(
    vocab_size=vocab_size,
    embedding_size=embedding_size,
    sequence_len=sequence_len,
    head_count=head_count,
    layer_count=layer_count,
).to(device=device)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 56769


In [ ]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch].to(device=device)
    y_batch = y_train[batch].to(device=device)
    train_logits = model(input=x_batch)

    train_loss = nn.functional.cross_entropy(input=train_logits.view(-1, vocab_size), target=y_batch.view(-1))

    if step % 1000 == 0 or step == steps - 1:
        model.eval()
        with torch.no_grad():
            val_logits = model(input=x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits.view(-1, vocab_size), target=y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")
        model.train()
        
    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.369324684143066, val loss = 4.361491680145264
Step 1000: train loss = 1.9342129230499268, val loss = 2.006688117980957
Step 2000: train loss = 1.7471680641174316, val loss = 1.893894076347351
Step 3000: train loss = 1.6949658393859863, val loss = 1.8359694480895996
Step 4000: train loss = 1.6232337951660156, val loss = 1.8217401504516602
Step 4999: train loss = 1.5472362041473389, val loss = 1.8139721155166626


In [ ]:
# prediction
model.eval()
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long, device=device)

    while len(output) < max_output_length:
        logits = model(input=i)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long, device=device)

In [86]:
print(''.join(output))

ROMEO:
I have, all is tunsatesty she than head's
But varreply to condents here out my old,
Whomment in singelant, the bettcomence our sonst
Againshourse or a king will wail dance course
Engling dose comes your say a pay: doing shame hand,
He first me rone croatiouse guety in
Kie metter mine up of off Delive out guizen:
I remsen up to craviss tydon daign him here?
We he boute beens, why is thou thoughy yea, sir,
Alvie wearly,
Than is 'tis me are, of lird his sheal dreve!

LEO:
That have!
And York's have never is her,
My h?

QUEEN RLANT:
I
LEarr:
Nor kind and tenders will the light upor,
The sliftlys ine,
Hade his guing eyed slaws whilt they heart,
Not, tit the say a hang it dom awll shall not
We fore, and by the sainted in that brieving.

HENRY BOLINGBRLE:
So should too; king bestring as they giive shall have
I wich for with this we blody unto I in light
That daughtes? slee autches, my beter sclick,
Office thee lighters scroe-fist?

SORCODes SLord's of quicken:
And,
Will thene? There se